In [5]:
import pandas as pd
df=pd.read_csv('delhi-weather-aqi-2025.csv')
df.head()

,date_ist,time_ist,location,lat,lon,temp_c,humidity,pressure_mb,windspeed_kph,condition_text,description,aqi_index,pm2_5,pm10,co,no2
0,01/01/2025,0:00,Anand Vihar,28.6469,77.316,8.1,100,995.4,2.9,Mainly clear,WMO Code 1,197,185.8,188.6,1907,56.7
1,01/01/2025,1:00,Anand Vihar,28.6469,77.316,7.7,100,994.7,3.2,Overcast,WMO Code 3,198,174.6,177.4,1669,44.8
2,01/01/2025,2:00,Anand Vihar,28.6469,77.316,7.5,100,994.3,4.5,Overcast,WMO Code 3,199,164.4,166.7,1493,34.6
3,01/01/2025,3:00,Anand Vihar,28.6469,77.316,7.8,99,994.1,6.0,Overcast,WMO Code 3,200,156.5,158.8,1401,26.7
4,01/01/2025,4:00,Anand Vihar,28.6469,77.316,7.3,100,993.8,6.8,Overcast,WMO Code 3,200,149.5,151.8,1372,20.6


In [6]:
df=df[['temp_c', 'humidity', 'pressure_mb', 'windspeed_kph', 'pm2_5', 'pm10', 'co','aqi_index']]


In [7]:
df.describe()

,temp_c,humidity,pressure_mb,windspeed_kph,pm2_5,pm10,co,aqi_index
count,52560.000000,52560.000000,52560.000000,52560.000000,52560.000000,52560.000000,52560.000000,52560.000000
mean,25.024406,62.285940,983.162635,6.565820,89.542432,300.462700,863.844273,287.364212
std,7.618231,23.454666,6.688582,3.478066,55.363812,412.782899,600.845140,327.979524
min,5.700000,5.000000,966.100000,0.000000,6.000000,6.100000,146.000000,56.000000
25%,19.500000,44.000000,977.500000,4.000000,51.100000,80.600000,464.000000,152.000000
50%,26.500000,65.000000,983.400000,6.000000,76.400000,151.800000,664.000000,170.000000
75%,30.500000,82.000000,988.900000,8.700000,110.400000,325.000000,1078.000000,238.000000
max,43.800000,100.000000,998.800000,27.100000,443.300000,3263.400000,7432.000000,2742.000000


In [12]:
X=df.drop(columns=['aqi_index'])
y=df['aqi_index']

In [ ]:

UNSAFE_AQI_THRESHOLD = 200

y = (y > UNSAFE_AQI_THRESHOLD).astype(int)

print(f"Safe (AQI <= 200): {(y == 0).sum()} rows")
print(f"Unsafe (AQI > 200): {(y == 1).sum()} rows")

Safe (AQI <= 200): 35886 rows
Unsafe (AQI > 200): 16674 rows


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier,StackingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score,balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold,GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, fbeta_score
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
numeric_features=['temp_c','humidity','windspeed_kph','pressure_mb','pm2_5','pm10','co']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())])
model_pipeline_xgb = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_features)
    ])),
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss',learning_rate=0.01, n_estimators=50, max_depth=5))
])
#applying the model pipeline to the training data
model_pipeline_xgb.fit(X_train, y_train)
#applying the grid search to find the best hyperparameters for the model
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2]
}
f2_scorer = make_scorer(fbeta_score, beta=2, average='weighted')
grid_search = GridSearchCV(model_pipeline_xgb, param_grid, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search.fit(X_train, y_train)

c:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:158: UserWarning: [12:16:56] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:158: UserWarning: [12:17:09] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [19]:
#evaluating the model on the test set
y_pred = grid_search.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.96      0.95      7263
           1       0.90      0.85      0.88      3249

    accuracy                           0.93     10512
   macro avg       0.92      0.91      0.91     10512
weighted avg       0.93      0.93      0.93     10512

Confusion Matrix:
 [[6967  296]
 [ 477 2772]]
Balanced Accuracy Score: 0.9062155432059354


In [20]:
# training by AdaBoostClassifier
model_pipeline_ada = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42,algorithm='deprecated'))
])
model_pipeline_ada.fit(X_train, y_train)
#finding the best hyperparameters for AdaBoostClassifier using grid search
param_grid_ada = {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__learning_rate': [0.01, 0.1, 1.0]
}
grid_search_ada = GridSearchCV(model_pipeline_ada, param_grid_ada, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_ada.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [21]:
#evaluating the AdaBoostClassifier model on the test set
y_pred_ada = grid_search_ada.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_ada))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ada))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_ada))

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.93      0.90      7263
           1       0.81      0.68      0.74      3249

    accuracy                           0.85     10512
   macro avg       0.84      0.81      0.82     10512
weighted avg       0.85      0.85      0.85     10512

Confusion Matrix:
 [[6746  517]
 [1034 2215]]
Balanced Accuracy Score: 0.8052827616771226


In [22]:
#training with GradientBoostingClassifier
model_pipeline_gb = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42))
])
model_pipeline_gb.fit(X_train, y_train)
param_grid_gb = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7],
    
}
grid_search_gb = GridSearchCV(model_pipeline_gb, param_grid_gb, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_gb.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [23]:
#evaluating the gradient boosting model on the test set
y_pred_gb = grid_search_gb.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_gb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gb))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_gb))


Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.97      0.95      7263
           1       0.92      0.87      0.89      3249

    accuracy                           0.94     10512
   macro avg       0.93      0.92      0.92     10512
weighted avg       0.94      0.94      0.94     10512

Confusion Matrix:
 [[7014  249]
 [ 419 2830]]
Balanced Accuracy Score: 0.9183769441212108


In [24]:
#training with random forest classifier
model_pipeline_rf = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42))
])
model_pipeline_rf.fit(X_train, y_train)
param_grid_rf = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
}
grid_search_rf = GridSearchCV(model_pipeline_rf, param_grid_rf, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__max_depth': [3, 5, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [25]:
#evalauting the random forest model on the test set
y_pred_rf = grid_search_rf.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_rf))

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.95      0.91      7263
           1       0.85      0.71      0.77      3249

    accuracy                           0.87     10512
   macro avg       0.87      0.83      0.84     10512
weighted avg       0.87      0.87      0.87     10512

Confusion Matrix:
 [[6865  398]
 [ 944 2305]]
Balanced Accuracy Score: 0.8273253842665534


In [26]:
#now training by StackingClassifier
base_estimators = [
    ('ada', AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42,algorithm='deprecated')),
    ('gb', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss',learning_rate=0.01, n_estimators=50, max_depth=5))
]
meta_estimator = RandomForestClassifier(max_depth=7, n_estimators=50, min_samples_split=5, min_samples_leaf=2,random_state=42)
stacking_model = StackingClassifier(estimators=base_estimators, final_estimator=meta_estimator, cv=5, n_jobs=-1)
stacking_model.fit(X_train, y_train)
param_grid_stacking = {
    'final_estimator__n_estimators': [50, 100, 200],
    'final_estimator__max_depth': [3, 5, 7],
}
grid_search_stacking = GridSearchCV(stacking_model, param_grid_stacking, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_stacking.fit(X_train, y_train)

,estimator,StackingClass... n_jobs=-1)
,param_grid,"{'final_estimator__max_depth': [3, 5, ...], 'final_estimator__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,estimator,None


In [27]:
#evaluating the stacking model on the test set
y_pred_stacking = grid_search_stacking.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_stacking))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_stacking))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_stacking))

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.94      0.93      7263
           1       0.85      0.81      0.83      3249

    accuracy                           0.90     10512
   macro avg       0.89      0.88      0.88     10512
weighted avg       0.90      0.90      0.90     10512

Confusion Matrix:
 [[6813  450]
 [ 609 2640]]
Balanced Accuracy Score: 0.8752999207076584


In [28]:
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve, recall_score, precision_score, fbeta_score

def get_optimal_fbeta_threshold_and_metrics(model, X_train, y_train, X_test, y_test, beta=2):
    """
    Finds the optimal threshold on training OOF probabilities by maximizing the F-beta score,
    then evaluates that custom threshold on the test set.
    """
    # 1. Get out-of-fold probabilities on the training set
    oof_probs = cross_val_predict(
        model, X_train, y_train, cv=5, method="predict_proba", n_jobs=-1
    )[:, 1]   

    # 2. Generate precision-recall curve pairs
    # we use unique thresholds to map out predictions efficiently
    precisions, recalls, thresholds = precision_recall_curve(y_train, oof_probs, pos_label=1)

    # 3. Find the best threshold using scikit-learn's fbeta_score directly
    fbeta_scores = []
    for thresh in thresholds:
        # Vectorized thresholding step
        train_preds_at_thresh = (oof_probs >= thresh).astype(int)
        # Using binary average to track your target 'fire_occurrence' class performance directly
        score = fbeta_score(y_train, train_preds_at_thresh, beta=beta, average='binary', zero_division=0)
        fbeta_scores.append(score)
        
    fbeta_scores = np.array(fbeta_scores)
    best_idx = np.argmax(fbeta_scores)
    best_threshold = thresholds[best_idx]
    train_expected_fbeta = fbeta_scores[best_idx]

    # 4. Evaluate this custom threshold on the TEST set
    test_probs = model.predict_proba(X_test)[:, 1]
    test_preds = (test_probs >= best_threshold).astype(int)
    
    test_recall = recall_score(y_test, test_preds, pos_label=1)
    test_precision = precision_score(y_test, test_preds, pos_label=1, zero_division=0)
    test_fbeta_result = fbeta_score(y_test, test_preds, beta=beta, average='binary', zero_division=0)

    return {
        "threshold": best_threshold,
        "train_expected_fbeta": train_expected_fbeta,
        "test_recall": test_recall,
        "test_precision": test_precision,
        "test_fbeta": test_fbeta_result
    }

# --- INTEGRATION LOOP ---

model_results = {
    "XGBoost": grid_search,
    "AdaBoost": grid_search_ada,
    "GradientBoosting": grid_search_gb,
    "RandomForest": grid_search_rf,
    "StackingClassifier": grid_search_stacking,
}

custom_metrics = {}
BETA_VALUE = 2  # Adjust this to change how much you prioritize recall over precision

print(f"Evaluating models using optimized F-{BETA_VALUE} thresholds:\n")

for name, fitted_search in model_results.items():
    best_est = fitted_search.best_estimator_
    
    # Run the generic F-beta threshold optimization function
    metrics = get_optimal_fbeta_threshold_and_metrics(best_est, X_train, y_train, X_test, y_test, beta=BETA_VALUE)
    custom_metrics[name] = metrics
    
    print(f"{name:20s} -> Best Thresh: {metrics['threshold']:.3f} | Train F{BETA_VALUE}: {metrics['train_expected_fbeta']:.4f} | Test Recall: {metrics['test_recall']:.4f} | Test Precision: {metrics['test_precision']:.4f}")

# Select the winner based on the highest achieved test F-beta score
best_model_name = max(custom_metrics, key=lambda k: custom_metrics[k]["test_fbeta"])
winner = custom_metrics[best_model_name]

print(f"\n>>> BEST MODEL BY OPERATIONAL F-{BETA_VALUE} SCORE: {best_model_name}")
print(f"    Operational Threshold: {winner['threshold']:.3f}")
print(f"    Final Test F-{BETA_VALUE} Score:  {winner['test_fbeta']:.4f}")
print(f"    Final Test Recall:     {winner['test_recall']:.4f}")
print(f"    Final Test Precision:  {winner['test_precision']:.4f}")

Evaluating models using optimized F-2 thresholds:

XGBoost              -> Best Thresh: 0.183 | Train F2: 0.9086 | Test Recall: 0.9551 | Test Precision: 0.7550
AdaBoost             -> Best Thresh: 0.433 | Train F2: 0.8123 | Test Recall: 0.8744 | Test Precision: 0.5863
GradientBoosting     -> Best Thresh: 0.159 | Train F2: 0.9154 | Test Recall: 0.9535 | Test Precision: 0.7757
RandomForest         -> Best Thresh: 0.206 | Train F2: 0.8332 | Test Recall: 0.8812 | Test Precision: 0.6461
StackingClassifier   -> Best Thresh: 0.182 | Train F2: 0.8705 | Test Recall: 0.9181 | Test Precision: 0.6913

>>> BEST MODEL BY OPERATIONAL F-2 SCORE: GradientBoosting
    Operational Threshold: 0.159
    Final Test F-2 Score:  0.9117
    Final Test Recall:     0.9535
    Final Test Precision:  0.7757


In [29]:
teacher_pipeline=grid_search_gb.best_estimator_
teacher_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [30]:
soft_labels = teacher_pipeline.predict_proba(X_train)

In [31]:
import tensorflow as tf
from tensorflow.keras import layers, models
student_model = models.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(soft_labels.shape[1], activation='softmax')
])
student_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
student_model.fit(X_train, soft_labels, epochs=50, batch_size=32, validation_split=0.2)

Epoch 1/50
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7935 - loss: 1.4922 - val_accuracy: 0.8183 - val_loss: 0.5694
Epoch 2/50
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8117 - loss: 0.7284 - val_accuracy: 0.8170 - val_loss: 0.9620
Epoch 3/50
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8117 - loss: 0.6770 - val_accuracy: 0.8442 - val_loss: 0.6623
Epoch 4/50
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8193 - loss: 0.6031 - val_accuracy: 0.8070 - val_loss: 0.5125
Epoch 5/50
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8248 - loss: 0.5516 - val_accuracy: 0.7950 - val_loss: 0.9776
Epoch 6/50
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8251 - loss: 0.5654 - val_accuracy: 0.7747 - val_loss: 0.9920
Epoch 7/50
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8344 - loss: 0.4736 - val_accuracy: 0.7980 - val_loss: 0.6279
Epoch 8/50
1052/1052 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8377 - loss: 0.4514 - 

In [32]:
convertor=tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_model = convertor.convert()

with open("AQI_predict_model_final.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\LOQ\AppData\Local\Temp\tmphf4xzgwi\assets


INFO:tensorflow:Assets written to: C:\Users\LOQ\AppData\Local\Temp\tmphf4xzgwi\assets


Saved artifact at 'C:\Users\LOQ\AppData\Local\Temp\tmphf4xzgwi'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 7), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2436605491712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2436605709824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2436605714576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2436605716864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2436605718096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2436605719504: TensorSpec(shape=(), dtype=tf.resource, name=None)
